# Collecte des segmentations par l'API Hugging Face

Ce notebook est le **seul** du projet qui appelle l'API. Il envoie chacune des 50 images au modèle
`sayeed99/segformer_b3_clothes`, deux fois : en JPEG réduit à 512 px (le réglage retenu) et en PNG d'origine
(pour vérifier que l'allègement ne coûte rien en qualité), plus une troisième fois en JPEG pleine taille, pour
isoler l'effet de la seule résolution (annexe du corrigé). Chaque carte prédite est enregistrée dans
`data/processed/predictions/`, chaque appel dans un journal.

Ensuite, le corrigé relit ce cache : il se ré-exécute **sans token et sans crédit**, et donne toujours les mêmes
chiffres. Relancer ce notebook ne refait que les appels manquants.

Prérequis : un fichier `.env` à la racine du projet contenant `HF_TOKEN=hf_...` (voir `.env.example`).

In [1]:
from segvet import api, cache, io

pairs = io.load_pairs()
images = [image for image, _ in pairs]
missing = {v: sum(not cache.prediction_path(p, v).exists() for p in images) for v in cache.VARIANTS}
print(f"{len(images)} images ; prédictions manquantes par variante : {missing}")

50 images ; prédictions manquantes par variante : {'jpeg512': 0, 'png': 0, 'jpeg_pleine_taille': 0}


## 1. Vérifier le token

Une requête simple au service d'identité de Hugging Face : si le token est refusé, inutile d'aller plus loin.
Le token n'est **jamais affiché** ; seul le nom du compte l'est.

In [2]:
if any(missing.values()):
    token = api.load_token()
    print("Token valide pour le compte :", api.check_token(token))
else:
    print("Cache complet : aucun appel à faire, le token n'est pas nécessaire.")

Cache complet : aucun appel à faire, le token n'est pas nécessaire.


## 2. Collecter

Réglages du client (fixés dans le `PLAN.md`) : timeout de 30 s par requête, au plus 30 requêtes par minute,
4 nouvelles tentatives sur les erreurs passagères (429, 5xx, timeout), arrêt immédiat sur 401, 402, 403 ou 404.
L'ordre des variantes s'inverse d'une image à l'autre pour qu'elles subissent les mêmes conditions réseau.
Le journal distingue la durée de la requête (`seconds`) du temps total, attentes du limiteur comprises
(`total_seconds`).

In [3]:
if any(missing.values()):
    client = api.SegformerClient(token, timeout=30, max_per_minute=30, max_retries=4)
    cache.collect(client, images, variants=tuple(cache.VARIANTS))
logs = {variant: cache.read_log(variant) for variant in cache.VARIANTS}

for variant, log in logs.items():
    print(f"{variant:>8} : {len(log)} images, statuts {log['status'].value_counts().to_dict()}")

 jpeg512 : 50 images, statuts {200: 50}
     png : 50 images, statuts {200: 50}
jpeg_pleine_taille : 50 images, statuts {200: 50}


## 3. Ce que l'API renvoie en plus de la réponse

Les en-têtes `x-…` conservés au premier appel réussi (utiles pour le chiffrage) :

In [4]:
first = logs["jpeg512"].query("status == 200").iloc[0]
{k: v for k, v in first["headers"].items() if "request-id" not in k and "trace" not in k}

{'x-inference-provider': 'hf-inference', 'x-cache': 'Miss from cloudfront'}